# 04 - Maps, change and indicators
* **Option A (fast, recommended first):** train the hybrid's Random Forest inside Earth Engine and map 2019 + 2025.
* **Option B (full hybrid):** run the weighted ensemble over the exported stack in Colab.

Note: one model trained on 2025 labels is applied to 2019 imagery. Check a sample of 2019 points
visually before trusting the change numbers.

In [ ]:
# --- Setup (run once per Colab session) ---
!git clone -q https://github.com/veeresh0804/PrithviBodh.git 2>/dev/null || (cd PrithviBodh && git pull -q)
%cd /content/PrithviBodh
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/PrithviBodh'      # Earth Engine exports land here
import os; os.makedirs(DRIVE, exist_ok=True)

import sys; sys.path.insert(0, '.')
from prithvibodh.config import load_config
cfg = load_config()
cfg['gee_project'] = 'YOUR_EE_PROJECT_ID'         # <-- your free noncommercial project
from prithvibodh import gee_data as G
G.init(cfg)

In [ ]:
import pandas as pd
from prithvibodh.features import all_columns
from prithvibodh import baselines
df = pd.read_csv(f'{DRIVE}/points_table.csv')
cols = all_columns(cfg)
# Option A: Earth Engine Random Forest on all labelled points, all sensors
for y in cfg['years']:
    G.export_map(G.gee_random_forest_map(cfg, y, df, cols), f'landcover_{y}', cfg)
G.export_map(baselines.dynamic_world(cfg, G.aoi(cfg), cfg['train_year']),
             f"dynamic_world_{cfg['train_year']}", cfg)
# Indicator stack (NDVI, LST, water, terrain) on the same grid, for vegetation change and risk layers
for y in cfg['years']:
    G.export_indicator_stack(cfg, y)          # use crs=cfg['aoi']['crs'] if you map with Option B
print('Exports started - wait for them in the Earth Engine Tasks tab')

In [ ]:
# Option B: full hybrid map (first run G.export_stack(cfg, year) for each year, as in notebook 03)
import os
from prithvibodh import experiment, hybrid, inference
from prithvibodh.deep import TemporalModel
X = df[cols].to_numpy('float32'); y = df.label.to_numpy()
members = {n: make().fit(X, y) for n, make in experiment.hybrid_members(cfg).items() if 'SVM' not in n}
weights = hybrid.normalise_weights({'H1 RF': 1, 'H3 TempCNN': 1, 'H4 Transformer': 1})  # or paste weights from notebook 03
for yr in cfg['years']:
    os.system(f'gdalbuildvrt -q {DRIVE}/stack_{yr}.vrt {DRIVE}/stack_{yr}*.tif')
    stack = f'{DRIVE}/stack_{yr}.vrt'
    inference.predict_map(stack, f'{DRIVE}/landcover_{yr}.tif', cfg, members, weights)

In [ ]:
# Change map, transition matrix and indicators (maps must share one grid - true for Option A and for Option B)
import numpy as np, rasterio
from prithvibodh import change, indicators, rasters
y0, y1 = cfg['years']
m0, prof = rasters.read(f'{DRIVE}/landcover_{y0}.tif')
m1, _ = rasters.read(f'{DRIVE}/landcover_{y1}.tif')
px_ha = 0.01
tm = change.transition_matrix(m0, m1, cfg['classes'], px_ha)
tm.to_csv(f'{DRIVE}/transition_matrix.csv')
rasters.write(f'{DRIVE}/change_{y0}_{y1}.tif', change.change_map(m0, m1, len(cfg['classes'])), prof)
summary = indicators.summary(m0, m1, cfg['classes'], y0, y1, px_ha)
tm.round(1), summary

In [ ]:
# Vegetation condition change (NDVI) from the indicator stacks
from prithvibodh.gee_data import INDICATOR_BANDS, INDICATOR_SCALE
ind0 = rasters.read_bands(f'{DRIVE}/indicators_{y0}.tif', ['ndvi_mean'], INDICATOR_BANDS, INDICATOR_SCALE)
ind1 = rasters.read_bands(f'{DRIVE}/indicators_{y1}.tif', ['ndvi_mean'], INDICATOR_BANDS, INDICATOR_SCALE)
idx = {c: i for i, c in enumerate(cfg['classes'])}
veg_ids = [idx['tree_cover'], idx['cropland'], idx['grass_shrub']]
veg = np.isin(m0, veg_ids) & np.isin(m1, veg_ids)
d_ndvi = ind1['ndvi_mean'] - ind0['ndvi_mean']
summary['mean_ndvi_change_vegetated'] = indicators.ndvi_change(ind0['ndvi_mean'], ind1['ndvi_mean'], veg)
# save as uint8: 0-200 maps NDVI change -1..+1, 255 = not vegetated
ndvi_u8 = np.where(veg, np.clip(np.round((d_ndvi + 1) * 100), 0, 200), 255)
rasters.write(f'{DRIVE}/ndvi_change_{y0}_{y1}.tif', ndvi_u8, prof)
pd.Series(summary).to_csv(f'{DRIVE}/indicators.csv')
summary

In [ ]:
# Copy outputs into the repo folder used by the Streamlit app (then commit, or upload to the app)
!mkdir -p outputs && cp {DRIVE}/landcover_*.tif {DRIVE}/dynamic_world_*.tif {DRIVE}/change_*.tif {DRIVE}/ndvi_change_*.tif {DRIVE}/results.csv {DRIVE}/transition_matrix.csv {DRIVE}/indicators.csv outputs/ 2>/dev/null; ls -lh outputs